# 畳み込みを、自分の画像で動かす

畳み込み・ReLU・プーリングを、ライブラリに任せず **for ループで** 書きます。
そして **あなたの写真** に掛けます。

図解は決まった小さな格子でしか動かせません。ここでは実物の画像で、
「機械が見るとはどういうことか」を確かめます。

- **戻る**: [畳み込みとプーリング](https://manga-epoch.github.io/viewer/pub/epoch/arc2/figures_cnn.html) — 同じ内容をスライダで動かせます
- **必要なもの**: NumPy・matplotlib・Pillow（Colab の標準環境でそのまま動きます）
- 上から順に実行してください（Colab では `Shift + Enter`）。

## 0. 画像を用意する

**自分の画像を使う場合**: 下のセルを実行するとファイル選択が出ます（Colab のみ）。
自分で撮った写真など、使う権利のある画像にしてください。

**何も選ばなかった場合**: その場で合成した画像を使います（外部の素材は使いません）。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from PIL import Image

def synthetic(n=192):
    """外部素材を使わずに、輪郭・縞・なだらかな面がそろった画像を作る。"""
    y, x = np.mgrid[0:n, 0:n]
    img = np.full((n, n), 0.45)
    img[(x - n*0.32)**2 + (y - n*0.36)**2 < (n*0.17)**2] = 0.92     # 円
    img[int(n*0.60):int(n*0.86), int(n*0.14):int(n*0.52)] = 0.12    # 四角
    img += 0.22 * (np.sin(x / 3.0) > 0) * ((x > n*0.58) & (y > n*0.14) & (y < n*0.5))
    img += np.linspace(0, .18, n)[None, :]                          # ゆるい明暗
    return np.clip(img, 0, 1)

img = None
try:
    from google.colab import files                    # Colab のときだけ通る
    up = files.upload()
    if up:
        name = list(up)[0]
        img = np.asarray(Image.open(name).convert("L"), dtype=float) / 255.0
        print(f"{name} を読み込みました")
except Exception:
    pass

if img is None:
    img = synthetic()
    print("合成画像を使います")

# 大きすぎると for ループが遅いので長辺 256px に収める
if max(img.shape) > 256:
    s = 256 / max(img.shape)
    img = np.asarray(Image.fromarray((img*255).astype(np.uint8))
                     .resize((int(img.shape[1]*s), int(img.shape[0]*s))), float) / 255.0

print("形:", img.shape, " 値の範囲:", round(img.min(), 2), "〜", round(img.max(), 2))
plt.figure(figsize=(4.5, 4.5)); plt.imshow(img, cmap="gray", vmin=0, vmax=1)
plt.title(f"input {img.shape}"); plt.axis("off"); plt.show()

## 1. 画像は、数の格子

グレースケールの画像は、明るさ（0 = 黒、1 = 白）が並んだ 2 次元配列です。
左上の 8×8 を数字で覗きます。

In [ ]:
np.set_printoptions(precision=2, suppress=True, linewidth=120)
print(img[:8, :8])

## 2. 畳み込み

小さな格子（**カーネル**）を画像の上で滑らせ、重なった部分どうしを掛けて足します。

$$y[i,j]=\sum_{u=0}^{K-1}\sum_{v=0}^{K-1} x[i\cdot S+u,\; j\cdot S+v]\;k[u,v]$$

$K$ がカーネルの大きさ、$S$ がずらす幅（ストライド）です。
ライブラリを使わず、そのまま書きます。

In [ ]:
def conv2d(x, k, stride=1, pad=0):
    if pad:
        x = np.pad(x, pad, mode="edge")
    K = k.shape[0]
    H = (x.shape[0] - K) // stride + 1
    W = (x.shape[1] - K) // stride + 1
    out = np.zeros((H, W))
    for i in range(H):
        for j in range(W):
            patch = x[i*stride:i*stride+K, j*stride:j*stride+K]
            out[i, j] = np.sum(patch * k)          # 掛けて、足す。これだけ
    return out

# 3×3 を手で組んで、動きを確かめる
tiny = np.arange(25, dtype=float).reshape(5, 5)
kid  = np.zeros((3, 3)); kid[1, 1] = 1.0           # 中央だけ 1 = 何もしないカーネル
print("入力:\n", tiny)
print("\n恒等カーネルを通すと、真ん中の 3×3 がそのまま出る:\n", conv2d(tiny, kid))

### カーネルを取り替えると、拾うものが変わる

同じ演算のまま、格子の数字だけを変えます。

In [ ]:
kernels = {
    "vertical edge":   np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float),
    "horizontal edge": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], float),
    "blur":            np.ones((3, 3)) / 9.0,
    "sharpen":         np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], float),
}

fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("input"); axes[0].axis("off")
for ax, (name, k) in zip(axes[1:], kernels.items()):
    o = conv2d(img, k, pad=1)
    ax.imshow(o, cmap="gray"); ax.set_title(name); ax.axis("off")
    print(f"{name:16} 出力 {o.shape}  範囲 {o.min():+.2f} 〜 {o.max():+.2f}")
plt.tight_layout(); plt.show()

`vertical edge` は縦の輪郭だけ、`horizontal edge` は横の輪郭だけが残ります。
**同じ画像・同じ演算で、格子の数字が違うだけ**です。

畳み込み層の学習とは、この 9 個の数字をデータから決めることにほかなりません。
人が「縦の輪郭を見ろ」と教えるのではなく、そうすると都合がいいと機械が見つけます。

## 3. 出力の大きさ

パディング $P$、ストライド $S$、カーネル $K$ のとき、出力の一辺は

$$\left\lfloor\frac{W-K+2P}{S}\right\rfloor+1$$

になります。**式が実際と合うか、総当たりで確かめます。**

In [ ]:
W = img.shape[1]
ok = True
print(f"{'K':>2} {'S':>2} {'P':>2} | {'式':>5} {'実際':>5}")
for K in (3, 5, 7):
    for S in (1, 2, 3):
        for P in (0, 1, 2):
            k = np.ones((K, K)) / (K*K)
            actual = conv2d(img, k, stride=S, pad=P).shape[1]
            formula = (W - K + 2*P) // S + 1
            ok &= actual == formula
            print(f"{K:>2} {S:>2} {P:>2} | {formula:>5} {actual:>5}"
                  + ("" if actual == formula else "   ← 不一致"))
print("\n→ すべて一致。" if ok else "\n→ 不一致あり。")

ストライドを 2 にすると出力が約半分になります。**画像を縮めながら見る**ということで、
深い層ほど「広い範囲をまとめて見る」ようになる仕組みの一つです。

## 4. ReLU — 負を捨てる

$$\mathrm{ReLU}(z)=\max(0,z)$$

輪郭検出の出力には正と負の両方が出ます（明→暗と暗→明）。
ReLU を通すと**片側だけ**が残ります。

In [ ]:
edge = conv2d(img, kernels["vertical edge"], pad=1)
relu = np.maximum(0, edge)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
axes[0].imshow(edge, cmap="RdBu_r", vmin=-2, vmax=2); axes[0].set_title("conv (red=+, blue=-)")
axes[1].imshow(relu, cmap="gray"); axes[1].set_title("after ReLU")
axes[2].hist(edge.ravel(), bins=60, color="#B4493F", alpha=.6, label="conv")
axes[2].hist(relu.ravel(), bins=60, color="#6E7BA8", alpha=.6, label="ReLU")
axes[2].set_yscale("log"); axes[2].legend(); axes[2].set_title("distribution")
for a in axes[:2]: a.axis("off")
plt.tight_layout(); plt.show()
print(f"負の値: 変換前 {(edge < 0).mean()*100:.1f}%  →  変換後 {(relu < 0).mean()*100:.1f}%")

## 5. プーリング — 位置ずれに強くする

小さな窓ごとに代表値を 1 つ取ります。最大値なら max pooling、平均なら average pooling。

In [ ]:
def pool2d(x, size=2, mode="max"):
    H, W = x.shape[0] // size, x.shape[1] // size
    v = x[:H*size, :W*size].reshape(H, size, W, size)
    return v.max(axis=(1, 3)) if mode == "max" else v.mean(axis=(1, 3))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (t, o) in zip(axes, [("input", relu),
                             ("max pool 2x2", pool2d(relu, 2, "max")),
                             ("avg pool 2x2", pool2d(relu, 2, "avg"))]):
    ax.imshow(o, cmap="gray"); ax.set_title(f"{t}  {o.shape}"); ax.axis("off")
plt.tight_layout(); plt.show()

### 検算 — 「位置ずれに強い」は本当か

そう言われますが、確かめないと主張のままです。**画像を横にずらして**、
ずらす前との違いを測ります。

ここで測り方に注意が要ります。max pooling は窓の**最大値**を取るので、
出力の値そのものが元より大きくなります。そのまま引き算して比べると、
プーリングした側が不利になるだけで、何も分かりません。

そこで **自分の大きさで割った相対変化** を見ます。

$$\text{相対変化}=\frac{\overline{|f(\text{ずらした画像})-f(\text{元の画像})|}}{\overline{|f(\text{元の画像})|}}$$

In [ ]:
def shift(x, d):
    return np.roll(x, d, axis=1)

KV = kernels["vertical edge"]
base = np.maximum(0, conv2d(img, KV, pad=1))

print("値の大きさ（平均絶対値）")
print(f"  畳み込みのまま {np.abs(base).mean():.4f} / "
      f"pool 2x2 {np.abs(pool2d(base,2)).mean():.4f} / "
      f"pool 4x4 {np.abs(pool2d(base,4)).mean():.4f}")
print("  → プーリング後のほうが値が大きい。絶対差のままでは比較にならない\n")

def relative_change(f, sh):
    a, b = f(sh), f(base)
    return np.abs(a - b).mean() / (np.abs(b).mean() + 1e-12)

print(f"{'ずれ':>5} | {'畳み込みのまま':>14} {'max pool 2x2':>13} {'max pool 4x4':>13}")
for d in (1, 2, 3, 4, 8):
    sh = np.maximum(0, conv2d(shift(img, d), KV, pad=1))
    r0 = relative_change(lambda z: z, sh)
    r2 = relative_change(lambda z: pool2d(z, 2), sh)
    r4 = relative_change(lambda z: pool2d(z, 4), sh)
    mark = "  ← 窓より大きいずれ" if d > 4 else ""
    print(f"{d:>3}px | {r0:>14.3f} {r2:>13.3f} {r4:>13.3f}{mark}")

読み取れることは、教科書的な「プーリングは位置ずれに強い」より少し厳密です。

- **ずれが窓より小さいうち**（1〜2px に対し窓 4）は、プーリングを通すほど変化が小さい。
  同じ窓の中で最大値を取る位置が入れ替わるだけなので、出力が変わらない
- **ずれが窓を超えると**、効果が消えます。値が隣の窓へ移ってしまうので、
  プーリングしてもしなくても同じか、むしろ大きく変わることもある

つまり **「どんなずれにも強い」のではなく、「窓の中のずれを吸収する」**。
だから CNN は小さなプーリングを何度も重ねます。1 回で大きな窓を使うのではなく、
段を追うごとに少しずつ吸収する範囲を広げていくわけです。

代わりに失うものもあります。**どこにあったかの情報**です。
分類には都合がよく、位置を答える検出やセグメンテーションでは困る
（[どこに何がある？](https://manga-epoch.github.io/viewer/pub/epoch/arc2/figures_detect.html)）。

## 6. 積む

畳み込み → ReLU → プーリングを 1 組として、繰り返します。
**通るたびに小さくなり、残るものが絞られていきます。**

In [ ]:
stage = img.copy()
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(stage, cmap="gray"); axes[0].set_title(f"input {stage.shape}"); axes[0].axis("off")
for i, ax in enumerate(axes[1:], start=1):
    stage = pool2d(np.maximum(0, conv2d(stage, kernels["vertical edge"], pad=1)), 2)
    ax.imshow(stage, cmap="gray"); ax.set_title(f"block {i}  {stage.shape}"); ax.axis("off")
    print(f"block {i}: {stage.shape}  画素数 {stage.size:,}")
plt.tight_layout(); plt.show()
print(f"\n入力 {img.size:,} 画素 → 最後 {stage.size:,} 画素（約 {img.size/stage.size:.0f} 分の 1）")

実際の CNN は、各段でカーネルを 1 枚ではなく数十〜数百枚使い、
**その数字をすべてデータから学習します**。ここで手で置いた 9 個の数字が、
AlexNet では 6000 万個になります（Arc 2 第 4 話）。

## 次に

- **図解に戻る**: [畳み込みとプーリング](https://manga-epoch.github.io/viewer/pub/epoch/arc2/figures_cnn.html)
  / [2012 年に何が揃ったのか](https://manga-epoch.github.io/viewer/pub/epoch/arc2/figures_alexnet.html)
- **学習まで書く**: [nn_from_scratch.ipynb](https://colab.research.google.com/github/manga-epoch/viewer/blob/main/notebooks/nn_from_scratch.ipynb)

---

## 出典とライセンス

このノートブックは、マンガ **EPOCH — 時代の前夜** の④「書く」レイヤーです。
[EPOCH について](https://manga-epoch.github.io/viewer)

このノートブックは画像を同梱していません。読者が用意した画像か、実行時に合成した画像だけを扱います。

本ノートブックのコードは自由に改変して使えます。